# 171Yb clock-to-Rydberg UV laser power estimate

Goal: estimate the 301.9 nm UV beam power needed to drive the `|c> = |3P0, F=1/2, mF=-1/2>` to `|r> = |65 3S1, F=3/2, mF=-3/2>` transition used in Muniz et al., PRX Quantum 6, 020334 (2025), DOI `10.1103/PRXQuantum.6.020334`.

Paper inputs used here:

- UV wavelength: 301.9 nm.
- Target Rydberg state: `|65 3S1, F=3/2, mF=-3/2>`.
- UV beam: approximately circular Gaussian beam with `w = 18 um` waist at the atoms.
- Reported system capability: `Omega_ryd > 2 pi x 15 MHz`; the repo's calibrated model often uses `Omega/(2 pi) = 10 MHz` as a conservative working value.

Important convention: this notebook estimates the optical power in the Gaussian beam at the atoms. It does not include upstream optical loss, AOM efficiency, imperfect polarization, clipping, or alignment margins.

## Formula

From `171yb_laser_control_theory_primer.md`, for a single-photon electric-dipole transition at the beam center,

$$\Omega = \frac{|d_{\rm eff}|}{\hbar}\sqrt{\frac{4P}{\pi w_x w_y c\epsilon_0}}.$$

Solving for power,

$$P = \frac{\pi w_xw_y c\epsilon_0\hbar^2}{4|d_{\rm eff}|^2}\Omega^2.$$

For the paper's approximately circular UV beam, `w_x = w_y = 18 um`. `rydcalc` returns the dipole matrix element in atomic units `e a0`, so the conversion to SI is `d_SI = d_au * e * a0`.

In [1]:
from __future__ import annotations

import math
import os
import sys
from pathlib import Path

from scipy import constants as cs

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

mpl_cache = Path("/tmp/matplotlib-rydcalc")
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

from neutral_yb.external.rydcalc_adapter import build_yb171_atom, has_arc_c_extension

print(f"project root: {root}")
print(f"rydcalc ARC C extension available: {has_arc_c_extension()}")

project root: /home/eris/projects/Noise-Tolerant-Quantum-Control-Optimization
rydcalc ARC C extension available: True


## Rydcalc matrix element

The UV transition is `sigma-`, because it changes `mF` from `-1/2` to `-3/2`. The `qIn=(-1,)` argument restricts the electric-dipole matrix element to that polarization component.

In [2]:
yb = build_yb171_atom(use_db=False)

clock_state = yb.get_state((6, 1, 1, 0, 0.5, -0.5), tt="NIST")
rydberg_state = yb.get_state((65, 0, 1.5, -1.5), tt="vlfm")

d_au = yb.get_multipole_me(clock_state, rydberg_state, qIn=(-1,))
d_q0 = yb.get_multipole_me(clock_state, rydberg_state, qIn=(0,))
d_qp = yb.get_multipole_me(clock_state, rydberg_state, qIn=(1,))
d_si = abs(d_au) * cs.e * cs.physical_constants["Bohr radius"][0]

print(clock_state)
print(rydberg_state)
print(f"d_eff(sigma-) = {d_au:.12g} e a0")
print(f"d_eff SI      = {d_si:.12e} C m")
print(f"sanity: q=0 -> {d_q0:.3g}, q=+1 -> {d_qp:.3g}")

|171Yb:6,S=1,L=1,j=0,F=0.5,-0.5>
|171Yb:64.56,L=0,F=1.5,-1.5>
d_eff(sigma-) = -0.00158576528275 e a0
d_eff SI      = 1.344467882514e-32 C m
sanity: q=0 -> 0, q=+1 -> 0


In [3]:
w_x = 18e-6
w_y = 18e-6

def gaussian_power_for_rabi(rabi_hz: float, d_si: float, w_x: float, w_y: float) -> dict[str, float]:
    omega = 2.0 * math.pi * rabi_hz
    power_w = math.pi * w_x * w_y * cs.c * cs.epsilon_0 * cs.hbar**2 * omega**2 / (4.0 * d_si**2)
    intensity_w_m2 = 2.0 * power_w / (math.pi * w_x * w_y)
    electric_field_v_m = cs.hbar * omega / d_si
    return {
        "rabi_hz": rabi_hz,
        "omega_rad_s": omega,
        "power_w": power_w,
        "intensity_w_m2": intensity_w_m2,
        "electric_field_v_m": electric_field_v_m,
    }

for result in [
    gaussian_power_for_rabi(10e6, d_si, w_x, w_y),
    gaussian_power_for_rabi(15e6, d_si, w_x, w_y),
]:
    print(
        f"Omega/(2pi) = {result['rabi_hz']/1e6:>4.1f} MHz: "
        f"P = {result['power_w']:.6f} W = {1e3*result['power_w']:.2f} mW, "
        f"I0 = {result['intensity_w_m2']:.3e} W/m^2, "
        f"E0 = {result['electric_field_v_m']:.3e} V/m"
    )

Omega/(2pi) = 10.0 MHz: P = 0.164065 W = 164.06 mW, I0 = 3.224e+08 W/m^2, E0 = 4.928e+05 V/m
Omega/(2pi) = 15.0 MHz: P = 0.369146 W = 369.15 mW, I0 = 7.253e+08 W/m^2, E0 = 7.393e+05 V/m


## Result

Using the `rydcalc` matrix element above and the paper's `18 um` beam waist:

- `Omega/(2 pi) = 10 MHz` requires about `0.164 W` at the atoms.
- `Omega/(2 pi) = 15 MHz` requires about `0.369 W` at the atoms.

The second number is the direct estimate for the paper's stated `Omega_ryd > 2 pi x 15 MHz` UV-system capability. Since power scales as `Omega^2` and as `w_x w_y`, other assumptions can be rescaled by

$$P \approx 0.369\,\mathrm{W}\left(\frac{\Omega/2\pi}{15\,\mathrm{MHz}}\right)^2\left(\frac{w_xw_y}{(18\,\mu\mathrm{m})^2}\right)\left(\frac{0.0015857653\,ea_0}{|d_{\rm eff}|}\right)^2.$$

Caveat: this is an ab-initio atomic-physics estimate, not a reconstruction of the lab's delivered laser power. The paper reports the beam waist and achievable Rabi rate, but not a full optical power budget at the UV source.